# Rendering showcase

A guided tour of `synthetic_scenes.render.render_scene`. Use the synthetic-scenes environment; examples are deterministic, require no GPU and do not download assets. Displayed images retain their rendered pixels, although the notebook frontend may scale them.

## Quick start

The specification is the request. The resolved scene records the request and actual layout when `scene` detail is requested; annotations describe the actual image in physical coordinates. Boxes are half-open `(left, top, right, bottom)`.

In [ ]:
from copy import deepcopy
from pathlib import Path
import os
from pprint import pprint
from IPython.display import display
from PIL import Image, ImageDraw
from synthetic_scenes.render import render_scene
from synthetic_scenes.augmentation import augment_scene
from synthetic_scenes.desktop_rendering.screenshot import _work_area

specification = {
    'mode': 'game', 'seed': 1,
    'windows': [{'type': 'minesweeper', 'content': {
        'rows': 9, 'columns': 9, 'mine_count': 10, 'status': 'active',
        'progress': .4, 'seed': 42, 'appearance': {'style': 'classic'},
    }}],
}
image, resolved, annotations = render_scene(specification, details=('scene',))
display(image)
pprint(resolved)
pprint(annotations['boards'][0])

## Presentation helpers

These helpers only assemble explicit examples and display existing renderer output. Each call creates an independent specification.

In [ ]:
def game(style='classic', **changes):
    content = deepcopy(specification['windows'][0]['content'])
    content['appearance'] = {'style': style, 'cell_size': 20}
    content.update(changes)
    return content

def standalone(content, mode='game'):
    return {'mode': mode, 'seed': 10, 'windows': [{'type': 'minesweeper', 'content': deepcopy(content)}]}

def desktop(windows, shell='taskbar', style='modern', scheme='light', scale=1., **options):
    return {'mode': 'desktop', 'seed': 11, 'layout_seed': 12,
            'screen': {'width': 960, 'height': 640, 'scale': scale},
            'desktop': {'shell': shell, 'style': style, 'color_scheme': scheme, **options},
            'wallpaper': {'category': 'procedural', 'subtype': 'linear', 'seed': 13},
            'windows': deepcopy(windows)}

def show(label, request, details=(), wallpaper_assets=None):
    image, resolved, annotations = render_scene(request, details=details, wallpaper_assets=wallpaper_assets)
    print(label, 'image:', image.size)
    display(image)
    return image, resolved, annotations

def outline(image, box, color):
    copy = image.copy()
    l, t, r, b = box
    ImageDraw.Draw(copy).rectangle((l, t, r-1, b-1), outline=color, width=2)
    return copy

## Game appearances and themes

The board seed and state stay fixed. Windows 7 `default` and `blue` are aliases. Each row uses the public grid mode.

In [ ]:
themes = {'classic': ('default', 'blue', 'green', 'warm'),
          'windows7': ('default', 'blue', 'green'),
          'microsoft': ('default', 'blue', 'warm'),
          'google': ('default', 'blue', 'warm')}
for style, names in themes.items():
    for theme in names:
        content = game(style)
        content['appearance']['theme'] = theme
        show(f'{style}: {theme}', standalone(content, 'grid'))
for variant in ('classic', 'flower'):
    content = game('windows7', status='lost', progress=.7)
    content['appearance']['mine_variant'] = variant
    show(f'Windows 7 mine: {variant}', standalone(content, 'grid'))
for boundary in (0., .1):
    content = game('google')
    content['appearance']['boundary_ratio'] = boundary
    show(f'Google boundary: {boundary}', standalone(content, 'grid'))

## Game states and typography

Progress is approximate, not an exact revealed-safe-cell fraction or a reconstruction of player history. Fonts resolve from the local system.

In [ ]:
for status, progress in [('new', 0.), ('active', .1), ('active', .45), ('active', .85), ('won', 1.), ('lost', .7)]:
    show(f'{status}, progress {progress}', standalone(game(status=status, progress=progress)))
for family, scale, bold in [('Arial', .72, True), ('Tahoma', .65, False), ('Verdana', .85, True)]:
    content = game(generation_mode='targeted', target_number=5)
    content['appearance'].update(font_family=family, font_scale=scale, bold=bold)
    show(f'{family}, scale {scale}, bold {bold}', standalone(content, 'grid'))
content = game(generation_mode='targeted', target_number=8)
content['appearance']['digit_palette'] = ('#005a6b', '#176a3d', '#a23420', '#543a86', '#8a3c58', '#146970', '#30245b', '#8d234b')
show('Custom digit palette', standalone(content, 'grid'))

## Targeted generation

Targets 1–8 are supported for active, won and lost games. The designated number is visible and the exact mine count is preserved. Progress is approximate. A structurally valid but infeasible target raises `ValueError`; invalid fields raise a validation error. Crops below use the returned cell box and nearest-neighbor magnification.

In [ ]:
for number in range(1, 9):
    request = standalone(game(generation_mode='targeted', target_number=number), 'grid')
    image, _, annotations = render_scene(request, details=('cell',))
    board = annotations['boards'][0]
    row, column = board['target_position']
    assert board['generation_mode'] == 'targeted' and board['target_number'] == number
    assert board['cell_states'][row][column] == number
    crop = image.crop(board['cell_boxes'][row][column])
    print('target', number, 'board position', (row, column))
    display(crop.resize((100, 100), Image.Resampling.NEAREST))

## Scene modes

The same content is rendered as a standalone grid, a complete game, and a desktop with external decorations and shell components.

In [ ]:
content = game('google')
for mode in ('grid', 'game', 'desktop'):
    request = desktop([{'type': 'minesweeper', 'content': content}]) if mode == 'desktop' else standalone(content, mode)
    image, resolved, annotations = show(mode, request, details=('scene',))
    print('layout:', resolved['layout'], 'boxes:', annotations['boards'][0]['boxes'])

## Desktop shells, appearance and decorations

Desktop appearance is independent of game appearance. Modern shells have shell-specific caption controls; classic windows use rectangular controls. These are complete desktop scenes.

In [ ]:
window = [{'type': 'minesweeper', 'content': game('google')}]
for shell, style, scheme, accent in [('taskbar', 'modern', 'light', '#4572B8'),
                                      ('taskbar', 'classic', 'dark', '#945044'),
                                      ('dock', 'modern', 'dark', '#417D71'),
                                      ('top_panel', 'modern', 'light', '#8057A0')]:
    show(f'{shell}, {style}, {scheme}', desktop(window, shell, style, scheme, accent_color=accent))

## Wallpapers

Procedural subtypes require no assets. To try a local image with cover, fit and repeat, set `SHOWCASE_WALLPAPER` to an existing image path before starting the kernel. No asset getter or download is used.

In [ ]:
for subtype in ('solid', 'linear', 'radial', 'multi_gradient'):
    request = desktop([])
    request['wallpaper']['subtype'] = subtype
    show(f'Procedural {subtype}', request)
asset = Path(os.environ.get('SHOWCASE_WALLPAPER', '')).expanduser()
if os.environ.get('SHOWCASE_WALLPAPER') and asset.is_file():
    for placement in ('cover', 'fit', 'repeat'):
        request = desktop([])
        request['wallpaper'] = {'category': 'local', 'seed': 13, 'placement': placement}
        show(f'Local wallpaper: {placement}', request, wallpaper_assets={'local': [asset]})
else:
    print('Optional asset examples skipped: set SHOWCASE_WALLPAPER to a local image.')

## Native zoom and display scaling

Base cell size and zoom control native game geometry. Desktop display scale changes native UI dimensions while the physical screenshot stays 960 × 640. Requested and resolved cell sizes and actual pitch come from renderer annotations.

In [ ]:
for zoom in (1., 1.5):
    content = game(); content['appearance']['zoom'] = zoom
    image, _, ann = render_scene(standalone(content), details=('scene',))
    b = ann['boards'][0]
    print('game zoom', zoom, 'image', image.size, 'requested/resolved', b['requested_cell_size'], b['resolved_cell_size'], 'pitch', b['style_parameters']['pitch'])
    display(image)
for scale in (1., 1.25):
    content = game(); content['appearance']['zoom'] = 1.25
    request = desktop([{'type': 'minesweeper', 'content': content}], scale=scale)
    image, _, ann = render_scene(request)
    b = ann['boards'][0]
    print('display scale', scale, 'game zoom 1.25, image', image.size, 'requested/resolved', b['requested_cell_size'], b['resolved_cell_size'], 'pitch', b['style_parameters']['pitch'])
    display(image)

## Component visibility and dock height

The reported work area comes from the renderer's own geometry. Hiding a component releases its reserved space. The dock has a four-logical-pixel bottom margin and no additional work-area separation.

In [ ]:
cases = [('taskbar', {'show_taskbar': True}), ('taskbar', {'show_taskbar': False}),
         ('dock', {'show_dock': True, 'show_status_bar': True, 'dock_height': 40}),
         ('dock', {'show_dock': False, 'show_status_bar': True}),
         ('dock', {'show_dock': True, 'show_status_bar': False, 'dock_height': 100}),
         ('top_panel', {'show_top_panel': True}), ('top_panel', {'show_top_panel': False})]
for shell, options in cases:
    request = desktop(window, shell=shell, **options)
    print(shell, options, 'work area', _work_area((960, 640), request['desktop'], 1.))
    show('components', request)

## Complete scenes and explicit gallery

Applications are placed behind protected Minesweeper windows, and shell components remain visible. These six specifications replace the former random sampler gallery; layout seeds are fixed.

In [ ]:
a = game('classic')
b = game('windows7', generation_mode='targeted', target_number=8, seed=55)
c = game('microsoft'); c['appearance']['zoom'] = 1.25
gallery = [
    ('Classic grid', standalone(a, 'grid')),
    ('Targeted Windows 7 game', standalone(b)),
    ('Modern desktop and browser', desktop([{'type': 'browser', 'seed': 3}, {'type': 'minesweeper', 'content': a}])),
    ('Classic desktop and two games', desktop([{'type': 'file_manager', 'seed': 4}, {'type': 'minesweeper', 'content': a}, {'type': 'minesweeper', 'content': b}], style='classic', scheme='dark')),
    ('Scaled dock desktop', desktop([{'type': 'minesweeper', 'content': c}], shell='dock', scheme='dark', scale=1.25, show_status_bar=False, dock_height=60)),
    ('Negative top-panel desktop', desktop([{'type': 'spreadsheet', 'seed': 6}], shell='top_panel', show_top_panel=False)),
]
for label, request in gallery:
    show(label, request, details=('scene',))

## Annotation details

`scene`, `cell`, and `mine` are independent options. Target and mine positions are board-relative `(row, column)` indices; cell and named boxes are image-space half-open rectangles. Mine positions describe the underlying minefield, including mines that are not visible. Diagnostic marks are drawn on a copy.

In [ ]:
request = standalone(game(generation_mode='targeted', target_number=8))
for detail in ('scene', 'cell', 'mine'):
    _, resolved, ann = render_scene(request, details=(detail,))
    print(detail, 'resolved keys:', list(resolved), 'board keys:', list(ann['boards'][0]))
image, resolved, ann = render_scene(request, details=('scene', 'cell', 'mine'))
b = ann['boards'][0]
diagnostic = image.copy(); draw = ImageDraw.Draw(diagnostic)
for name, color in [('game_region', 'yellow'), ('grid', 'cyan')]:
    l,t,r,bottom = b['boxes'][name]; draw.rectangle((l,t,r-1,bottom-1), outline=color, width=2)
for row in b['cell_boxes']:
    for l,t,r,bottom in row:
        draw.rectangle((l,t,r-1,bottom-1), outline='#777777')
for row, col in b['mine_positions']:
    l,t,r,bottom = b['cell_boxes'][row][col]
    draw.ellipse((l+4,t+4,r-4,bottom-4), outline='red', width=2)
row, col = b['target_position']
l,t,r,bottom = b['cell_boxes'][row][col]
draw.rectangle((l,t,r-1,bottom-1), outline='magenta', width=3)
print('target', b['target_position'], 'visible state', b['cell_states'][row][col], 'underlying mines', b['mine_positions'])
display(diagnostic)

## Capture augmentation

Augmentation applies resize and capture effects to the image and transforms the same image-space annotations. The overlay uses the returned grid box, on a copy.

In [ ]:
image, _, ann = render_scene(standalone(game('microsoft', generation_mode='targeted', target_number=7)), details=('cell',))
for label, options in [('original', {}), ('resize', {'resize_factor': .75}),
                       ('brightness and contrast', {'brightness': 1.1, 'contrast': .9}),
                       ('blur and JPEG', {'blur': .5, 'jpeg_quality': 70})]:
    adjusted, adjusted_ann = augment_scene(image, ann, **options)
    print(label, adjusted.size, adjusted_ann['boards'][0]['boxes']['grid'])
    display(outline(adjusted, adjusted_ann['boards'][0]['boxes']['grid'], 'cyan'))